# 11.3 生成增强 (Generation Enhancement)

检索到文档后，如何有效利用检索结果增强生成质量。

本节涵盖：上下文注入、引用生成、自适应检索

## 1. 上下文注入与引用生成

**上下文注入**：将检索到的文档作为上下文拼接到提示中，让模型基于检索内容生成回答。

**引用生成**：模型生成回答时标注信息来源，增强可信度和可验证性。

**提示模板**：
```
Based on the following documents, answer the question.
Documents: [doc1] [doc2] ...
Question: [query]
Answer with citations: [1], [2] etc.
```

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

class RAGGenerator(nn.Module):
    def __init__(self, d=64, vocab_size=100):
        super().__init__()
        self.context_proj = nn.Linear(d, d)
        self.query_proj = nn.Linear(d, d)
        self.output = nn.Linear(d, vocab_size)

    def forward(self, query, contexts, context_mask=None):
        q = self.query_proj(query)
        c = self.context_proj(contexts)
        if context_mask is not None:
            attn = F.softmax(q @ c.transpose(-2, -1) / (q.shape[-1] ** 0.5), dim=-1)
            attn = attn * context_mask.unsqueeze(1)
            attn = attn / (attn.sum(dim=-1, keepdim=True) + 1e-8)
        else:
            attn = F.softmax(q @ c.transpose(-2, -1) / (q.shape[-1] ** 0.5), dim=-1)
        context_repr = (attn @ c).mean(dim=1)
        combined = q.squeeze(1) + context_repr
        return self.output(combined), attn

generator = RAGGenerator()
query = torch.randn(1, 1, 64)
contexts = torch.randn(1, 5, 64)
logits, attn_weights = generator(query, contexts)

print('=== RAG Generation ===')
print(f'Query: {query.shape}, Contexts: {contexts.shape}')
print(f'Output logits: {logits.shape}')
print(f'Attention over contexts: {attn_weights[0, 0].detach().tolist()}')
most_relevant = attn_weights[0, 0].argmax().item()
print(f'Most relevant context: [{most_relevant + 1}]')
print(f'\nKey: Attention weights show which context documents are most relevant.')
print(f'These can be used as citation markers in the generated answer.')

## 2. 自适应检索 (Adaptive Retrieval)

**FLARE**：模型在生成过程中主动判断何时需要检索，只在不确定时才检索。

**Self-RAG**：模型自我反思检索和生成的质量，决定是否需要重新检索。

**核心思想**：不是每次都检索，而是让模型自己决定何时检索、检索什么。

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

class AdaptiveRetrievalDecider(nn.Module):
    def __init__(self, d=64):
        super().__init__()
        self.confidence_net = nn.Sequential(
            nn.Linear(d, 32), nn.ReLU(), nn.Linear(32, 1), nn.Sigmoid()
        )

    def forward(self, hidden_state, threshold=0.5):
        confidence = self.confidence_net(hidden_state)
        need_retrieval = confidence < threshold
        return confidence, need_retrieval

decider = AdaptiveRetrievalDecider()
states = torch.randn(8, 64)
confidences, decisions = decider(states)

print('=== Adaptive Retrieval ===')
print(f'Confidence scores: {confidences.squeeze().detach().tolist()}')
print(f'Need retrieval: {["Yes" if d else "No" for d in decisions.squeeze().tolist()]}')
n_retrieve = decisions.sum().item()
print(f'Retrieval triggered: {n_retrieve}/{len(states)} steps ({n_retrieve/len(states):.0%})')
print(f'\nKey: Adaptive retrieval only triggers when model is uncertain.')
print(f'Saves computation and reduces irrelevant context injection.')